# Results & Analysis: Maize Disease CNN Experiment 1 vs 2

This document chronicles the official thesis findings from building a Custom CNN and testing the hypothesis: **Does standard Computer Vision preprocessing (OpenCV hue masking, morphology, CLAHE) improve the predictive power of a Deep Learning model?**


## 1. Experiment 1: The Baseline (Raw Images)
**Architecture:** Custom baseline CNN
**Data:** Unaltered Keras train/test splits.

### Held-out Test Metrics:
The model achieved an outstanding **93.69% overall accuracy** on the 1,173 raw test images.
```text
Confusion matrix (rows=true, cols=pred):
[[296   4  27   0]
 [ 13 348   2   1]
 [ 25   0 136   0]
 [  0   0   2 319]]

                precision    recall  f1-score   support

        Blight     0.88      0.91      0.90       327
   Common_Rust     0.99      0.96      0.97       364
Gray_Leaf_Spot     0.81      0.84      0.83       161
       Healthy     1.00      0.99      1.00       321

      accuracy                         0.94      1173
```


## 2. Experiment 2: OpenCV Preprocessed Images
**Architecture:** EXACT same Custom baseline CNN
**Data:** OpenCV preprocessed images (Gaussian blur, HSV masking, Morphology cleaning, CLAHE).

### Held-out Test Metrics:
The metric accuracy **DROPPED to 91.82%** on the preprocessed test set.
```text
Confusion matrix (rows=true, cols=pred):
[[270  12  45   0]
 [ 12 348   3   1]
 [ 18   4 139   0]
 [  0   1   0 320]]

                precision    recall  f1-score   support

        Blight     0.90      0.83      0.86       327
   Common_Rust     0.95      0.96      0.95       364
Gray_Leaf_Spot     0.74      0.86      0.80       161
       Healthy     1.00      1.00      1.00       321

      accuracy                         0.92      1173
```


## 3. Experiment 3: ResNet50 (Transfer Learning) on Preprocessed Images
**Architecture:** ResNet50 (pre-trained on ImageNet, last 30 layers unfrozen)
**Data:** EXACT same OpenCV preprocessed dataset as Experiment 2.

### Held-out Test Metrics:
By utilizing Transfer Learning, the ResNet50 model decimated the previous records, achieving a staggering **96.25% overall accuracy**!
```text
Confusion matrix (rows=true, cols=pred):
[[307   7  12   1]
 [  4 357   3   0]
 [ 15   2 144   0]
 [  0   0   0 321]]

                precision    recall  f1-score   support

        Blight     0.94      0.94      0.94       327
   Common_Rust     0.98      0.98      0.98       364
Gray_Leaf_Spot     0.91      0.89      0.90       161
       Healthy     1.00      1.00      1.00       321

      accuracy                         0.96      1173
```

### Analysis of the ResNet50 Leap:
The results completely validate the power of deep transfer learning. While the Custom CNN (Experiment 2) lacked the depth to make sense of the harsh geometric boundaries left behind by the OpenCV background subtraction (resulting in a 91.82% score), the 23-million-parameter ResNet50 model effortlessly bypassed the missing background textures. Because it had already learned universal geometric feature extraction from ImageNet, it could purely evaluate the true disease characteristics on the leaf surface, boosting the complex `Gray_Leaf_Spot` F1-score from 0.80 all the way to 0.90.


## 3. Thesis Discussion: Why did preprocessing hurt performance?
While classical Computer Vision techniques like HSV masking and CLAHE normalization were designed to remove background noise and help algorithms focus on leaf geometry, the results clearly show that they disrupted the Deep Learning feature extractors. 

Deep Learning models thrive on immense, raw entropy. It is highly likely that illnesses like Blight and Gray Leaf Spot contain exceptionally subtle pixel textures, color gradients, or even background interactions that a CNN relies on space-wise. By intentionally zeroing out backgrounds and forcing uniform contrast via CLAHE, the dataset was stripped of fundamental variance that was secretly providing hidden correlation power to the CNN's latent space, hurting its ability to correctly identify the subtler classes (like Gray Leaf Spot dropping from an 0.83 to a 0.80 F1-score).
